# Entity velocity & acceleration — starter analysis

Loads the newest daily snapshot from `data/research_private/snapshots/`
and produces the top-mover / burst tables that back the paper's
velocity chapter. The private corpus is gitignored — you must have
run `run-research.bat` (or `python -m pipeline.research.snapshot`)
at least once locally for these cells to find data.

**Reproduction:** every metric here is defined in `research/methodology.md`
and computed by `pipeline/research/trend_metrics.py`. This notebook is
only a viewer — no analytic logic lives here.

In [ ]:
from pathlib import Path

import pandas as pd

# Notebook lives at research/notebooks/, so go up two levels to the repo root.
REPO_ROOT = Path.cwd().parents[1]
SNAPSHOTS = REPO_ROOT / "data" / "research_private" / "snapshots"
assert SNAPSHOTS.exists(), (
    f"No snapshots yet. Run `python -m pipeline.research.snapshot` first.\n"
    f"Expected: {SNAPSHOTS}"
)

latest = sorted(p for p in SNAPSHOTS.iterdir() if p.is_dir())[-1]
print(f"Latest snapshot: {latest.name}")

In [ ]:
velocity = pd.read_parquet(latest / "entity_velocity.parquet")
acceleration = pd.read_parquet(latest / "entity_acceleration.parquet")
bursts = pd.read_parquet(latest / "entity_bursts.parquet")
print(f"velocity rows: {len(velocity):,}")
print(f"acceleration rows: {len(acceleration):,}")
print(f"burst rows: {len(bursts):,}")
velocity.head()

## Top 10 fastest-growing entities (latest day in corpus)

Uses raw velocity `v(e, t) = M(e, t) - M(e, t-1)`.

In [ ]:
latest_day = velocity["day"].max()
print(f"Ranking day: {latest_day}")

top10 = (
    velocity[velocity["day"] == latest_day]
    .sort_values("velocity", ascending=False)
    .head(10)
)
top10[["entity", "entity_type", "velocity", "velocity_ema7"]]

## Top 10 fastest-decaying entities (latest day)

In [ ]:
bottom10 = (
    velocity[velocity["day"] == latest_day]
    .sort_values("velocity", ascending=True)
    .head(10)
)
bottom10[["entity", "entity_type", "velocity", "velocity_ema7"]]

## Kleinberg-style bursts (|z| >= 2)

Entities whose latest-day velocity is more than 2 standard deviations
from their own historical mean. These are the candidates for narrative
'moment' study in the paper.

In [ ]:
burst_today = bursts[(bursts["day"] == latest_day) & (bursts["burst_z"].abs() >= 2)]
burst_today.sort_values("burst_z", ascending=False)

## Velocity vs. acceleration scatter

Quadrant reading:
- **top-right** (v > 0, a > 0): still accelerating
- **top-left**  (v > 0, a < 0): growing but slowing (peak nearby)
- **bottom-right** (v < 0, a > 0): decaying but rebound starting
- **bottom-left** (v < 0, a < 0): decline compounding

In [ ]:
merged = velocity.merge(acceleration, on=["day", "entity", "entity_type"])
phase = merged[merged["day"] == latest_day].copy()
ax = phase.plot.scatter("velocity", "acceleration", alpha=0.6, figsize=(8, 6))
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
for row in phase.nlargest(5, "velocity").itertuples(index=False):
    ax.annotate(row.entity, (row.velocity, row.acceleration))
ax.set_title(f"Phase space on {latest_day}")

## Paper-level trends (Z2 join layer)

`pipeline/research/paper_trends.py` joins the private `papers.db`
with news tags so papers themselves become trend entities. Run
`python -m pipeline.research.paper_trends` (or `run-research.bat`)
first — outputs land under `data/research_private/paper_trends/`.

In [ ]:
import json

PT = REPO_ROOT / "data" / "research_private" / "paper_trends"
if not PT.exists():
    print("No paper_trends yet - run `python -m pipeline.research.paper_trends` first.")
else:
    pvel = pd.read_parquet(PT / "paper_velocity.parquet")
    ptop = pd.read_parquet(PT / "paper_topics.parquet")
    print(f"paper_velocity: {len(pvel):,} rows / {pvel['arxiv_id'].nunique()} papers")
    print(f"paper_topics:   {len(ptop):,} rows")

    hot_files = sorted(PT.glob("hot_papers-*.json"))
    if hot_files:
        hot = json.loads(hot_files[-1].read_text(encoding="utf-8"))
        print(f"\nhot papers (as_of {hot['as_of']}):")
        display(pd.DataFrame(hot["papers"]))

    # Most-tagged papers - which research topics the news corpus
    # attaches to papers most often.
    ptop.groupby("tag")["count"].sum().sort_values(ascending=False).head(10)